# WTI 2026 Eval — Does the Anchored Agent Beat Prophet?

`01_wti_case_study.ipynb` tells the story of Prophet forecasting blind through the
early-2026 Persian Gulf shock: its 95% CI caught ~77% of resolutions through 2025,
then the geopolitical spike broke it. `04_systematic_backtest_eval.ipynb` already
runs a full stateless leaderboard (Naive, AutoARIMA, LightGBM, Prophet, LLM/agent
methods) across the same 18-origin `energy_oil_eval` window and produces the
scorecard + per-origin panel chart this notebook reuses verbatim from `viz.py` /
`analysis.py` — nothing here is a new charting pipeline, it's the same one with two
more rows added: the real, live-run **anchored agent**
(`AnchoredAgentPredictor`, Day-1's guardrail deliverable, `w_loc`/`w_width` fit in
`anchor_weight_fitting.ipynb`), and a **mechanism-isolation ablation** built from
that run's own logged signals, at zero extra LLM cost.

Everything below is a **real measurement** — every row in every table and chart is
either a cached artifact under `data/predictions/energy_oil_eval/` or a live run of
`AnchoredAgentPredictor` persisted the same way. No projected/simulated numbers.

In [1]:
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import properscoring as ps
import yaml
from IPython.display import HTML, display  # noqa: A004

import energy_oil_forecasting
from aieng.forecasting.evaluation import (
    BacktestResult,
    ContinuousForecast,
    MultiTargetBacktestSpec,
    Prediction,
    cached_multi_backtest,
)
from aieng.forecasting.evaluation.artifacts import load_multi_backtest_results
from energy_oil_forecasting import viz
from energy_oil_forecasting.analysis import (
    build_price_frame,
    extract_agent_rationales,
    leaderboard_with_uncertainty,
    per_horizon_crps,
    predictions_to_frame,
    score_backtest_results,
)
from energy_oil_forecasting.analyst_agent.anchor_lookup import AnchorSource
from energy_oil_forecasting.analyst_agent.anchored_predictor import build_wti_anchored_predictor
from energy_oil_forecasting.analyst_agent.news_cache import NewsCacheSource
from energy_oil_forecasting.data import build_wti_service

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)

spec_dir = Path(energy_oil_forecasting.__file__).parent / "specs"
with open(spec_dir / "energy_oil_eval.yaml") as f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))
task = eval_spec.tasks[0]
single_spec = eval_spec.specs()[0]

data_service = build_wti_service()
price_df = build_price_frame(data_service)
anchor_source = AnchorSource.from_spec_id("energy_oil_eval")

_now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
_actuals = data_service.get_series(task.target_series_id, as_of=_now).copy()
_actuals["timestamp"] = pd.to_datetime(_actuals["timestamp"])


def _resolve(forecast_date) -> float | None:
    match = _actuals[_actuals["timestamp"] == pd.Timestamp(forecast_date)]
    return float(match["value"].iloc[0]) if not match.empty else None


def color_key_html(colors: dict[str, str]) -> str:
    # Plain HTML swatch legend -- a guaranteed-visible fallback next to any
    # Plotly chart whose own legend doesn't render in every notebook viewer.
    items = "".join(
        f"<span style='display:inline-flex;align-items:center;margin:2px 14px 2px 0'>"
        f"<span style='width:12px;height:12px;border-radius:2px;background:{c};"
        f"display:inline-block;margin-right:6px'></span>{name}</span>"
        for name, c in colors.items()
    )
    return f"<div style='font-size:13px;padding:4px 0'>{items}</div>"


## 1. Load the stateless + free-form baselines already evaluated on 2026

All cached artifacts under `data/predictions/energy_oil_eval/` from Notebook 4's
full registry run — no recomputation, no LLM calls, just reads.

In [2]:
EVAL_PREDICTOR_IDS = {
    "Naive (Last Value)": "last_value_naive",
    "Prophet": "prophet_daily",
    "AutoARIMA": "darts_autoarima",
    "AutoARIMA (seeded anchor)": "darts_autoarima_seed42_n500",
    "News Agent (preview, LIVE search baseline)": "agent_predictor_wti_analyst_news_gemini-3.1-flash-lite-preview_continuous",
    "News Agent (3.5-flash, LIVE search baseline)": "agent_predictor_wti_analyst_news_gemini-3.5-flash_continuous",
    # "cached" = reads the SAME NewsCacheSource briefings as the anchored agent
    # (see section 2b) -- this is the one the rebuild's before/after numbers in
    # planning-docs/news-cache-rebuild-interview-notes.md §11.2 refer to. It is a
    # DIFFERENT predictor_id from the two rows above (which use live search_web
    # and were never touched by the rebuild) -- do not conflate the two "News
    # Agent" families, they can differ by ~1 point of CRPS for reasons unrelated
    # to news quality (live search sees a different, larger set of articles).
    "News Agent (preview, CACHED news)": "agent_predictor_wti_analyst_news_cached_gemini-3.1-flash-lite-preview_continuous",
    "News Agent (3.5-flash, CACHED news)": "agent_predictor_wti_analyst_news_cached_gemini-3.5-flash_continuous",
    "Adaptive Agent (trained)": "agent_predictor_wti_adaptive_analyst_wti_strategy_continuous",
}

eval_results: dict[str, dict[str, BacktestResult]] = {}
for display_name, predictor_id in EVAL_PREDICTOR_IDS.items():
    results = load_multi_backtest_results(eval_spec, predictor_id)
    if results is None:
        print(f"  MISSING  {display_name} ({predictor_id})")
        continue
    eval_results[display_name] = results
    n = len(results[task.task_id].predictions)
    print(f"  loaded   {display_name:35s} n={n}")

  loaded   Naive (Last Value)                  n=22
  loaded   Prophet                             n=16
  loaded   AutoARIMA                           n=22
  loaded   AutoARIMA (seeded anchor)           n=50
  loaded   News Agent (preview, LIVE search baseline) n=50
  loaded   News Agent (3.5-flash, LIVE search baseline) n=50
  loaded   News Agent (preview, CACHED news)   n=50
  loaded   News Agent (3.5-flash, CACHED news) n=50
  loaded   Adaptive Agent (trained)            n=22


## 2. Run the real anchored agent — 18 origins, live LLM calls

The actual `AnchoredAgentPredictor` (bounded `signal_loc`/`signal_width` schema,
`w_loc=0.2`/`w_width=0.5` fit in `anchor_weight_fitting.ipynb`), run live across all
18 `energy_oil_eval` origins using the now-complete news cache.
`cached_multi_backtest` persists the result to `data/predictions/energy_oil_eval/`
and skips recomputation on any later re-run of this notebook — this cell makes real
LLM calls exactly once.

In [3]:
W_LOC = 0.2
W_WIDTH = 0.5

anchored_predictor = build_wti_anchored_predictor(
    anchor_source,
    news_source=NewsCacheSource(),
    w_loc=W_LOC,
    w_width=W_WIDTH,
)
print(f"predictor_id: {anchored_predictor.predictor_id}")

real_anchored_results = cached_multi_backtest(anchored_predictor, eval_spec, data_service)
eval_results["Anchored Agent (real run)"] = real_anchored_results
_real = real_anchored_results[task.task_id]
print(f"n={len(_real.predictions)}  mean_crps={_real.mean_score:.4f}")

predictor_id: agent_predictor_wti_analyst_anchored_cached_news_gemini-3.1-flash-lite-preview_continuous
n=50  mean_crps=9.1100


> **Sample-size caveat — RESOLVED 2026-08-09.** This section originally read: *"this
> run only resolved 14 of 18 origins (n=42, not 54)... worth fixing in anchor-table
> generation."* That fix has since landed —
> `AnchorSource.available_horizons()` + changes in `AnchoredWtiPromptBuilder` and
> `_reconstruct_predictions` (see `planning-docs/news-cache-rebuild-interview-notes.md`
> §"anchor-table holiday gap") — so a missing horizon (2026-02-16 is Presidents' Day,
> 2026-05-25 is Memorial Day; pandas business-day stepping doesn't know either one is
> closed) now only drops *that* horizon instead of the whole origin.
>
> This run (re-executed 2026-08-09, same `w_loc=0.2`/`w_width=0.5`, against the
> rebuilt news cache — see below) now resolves **n=50/54**, matching the raw
> baselines' own count almost exactly. The comparison in this notebook is no longer
> apples-to-oranges on sample size.</cell>


## 2b. Same mechanism, two prompt changes — testing a measured asymmetry

Section 2's run has a defect that only shows up when it is paired against the
*free-form* agent on identical `(origin, horizon, cached news)` cells: the bounded
schema does not shrink `signal_loc` symmetrically, it **rectifies** it. Where the
free-form agent forecast *below* the anchor, the anchored agent mostly did not
follow — it reported zero, or flipped positive. Pure shrinkage would scale both
sides by the same factor; measured pass-through ratios were `+0.55` up vs. `-0.37`
down (preview) and `+1.15` vs. `-0.08` (3.5-flash). The two agents agree on
*ranking* (Spearman +0.64 / +0.62, p<0.001), so this is not disagreement about the
market — the bottom half of the signal is lost in expression.

That matters for everything downstream, because in this window the truth landed
**above** the anchor 60% of the time. A signal that structurally cannot say "down"
scores well here *by construction* — so section 8's "the location signal earns its
keep" is confounded until this is understood.

Two rewrites of `_ANCHOR_SUPPLEMENT` are tested against the original, all three
kept as named variants in `ANCHOR_PROMPT_VARIANTS` (`agent.py`) so their
predictions persist to separate files and stay directly comparable — schema,
bounds, anchor table, news cache, and reconstruction arithmetic are identical
across all three; only the prompt wording/structure differs.

- **`symloc`** (`_ANCHOR_SUPPLEMENT_SYMMETRIC_LOC`) — framing only. In the
  original, `signal_width`'s bullet spends a full clause on *"there is no
  negative half"* while `signal_loc`'s negative half is only implied by interval
  notation. `symloc` separates the two signals and states `signal_loc`'s
  negative half as explicitly as `signal_width`'s absence of one — same
  vocabulary, same structure, just plainer about the one clause.
- **`twosided`** (`_ANCHOR_SUPPLEMENT_TWO_SIDED`) — structural. Forces a written
  bearish case before a number is chosen, and neutralises the one-directional
  "geopolitical risk" vocabulary in the Role line and rules 3-4 that `symloc`
  left untouched.

The original prompt is untouched and remains the default (`anchor_prompt="original"`).
`symloc`'s and `twosided`'s cached-news predictions were generated via
`scripts/run_cached_news_2x2.py` for both models — no new LLM calls happen in
this notebook to reproduce them, only cache reads.</cell id="3cf20445">

In [4]:
modified_prompt_predictor = build_wti_anchored_predictor(
    anchor_source,
    news_source=NewsCacheSource(),
    w_loc=W_LOC,
    w_width=W_WIDTH,
    anchor_prompt="symloc",
)
print(f"predictor_id: {modified_prompt_predictor.predictor_id}")

modified_prompt_results = cached_multi_backtest(modified_prompt_predictor, eval_spec, data_service)
eval_results["Anchored Agent (real run, symloc prompt)"] = modified_prompt_results
_modified = modified_prompt_results[task.task_id]
print(f"n={len(_modified.predictions)}  mean_crps={_modified.mean_score:.4f}")

predictor_id: agent_predictor_wti_analyst_anchored_cached_news_symloc_gemini-3.1-flash-lite-preview_continuous
n=50  mean_crps=9.0849


In [5]:
twosided_predictor = build_wti_anchored_predictor(
    anchor_source,
    news_source=NewsCacheSource(),
    w_loc=W_LOC,
    w_width=W_WIDTH,
    anchor_prompt="twosided",
)
print(f"predictor_id: {twosided_predictor.predictor_id}")

twosided_results = cached_multi_backtest(twosided_predictor, eval_spec, data_service)
eval_results["Anchored Agent (real run, twosided prompt)"] = twosided_results
_twosided = twosided_results[task.task_id]
print(f"n={len(_twosided.predictions)}  mean_crps={_twosided.mean_score:.4f}")

predictor_id: agent_predictor_wti_analyst_anchored_cached_news_twosided_gemini-3.1-flash-lite-preview_continuous
n=50  mean_crps=9.0876


In [6]:
# Pair each anchored run against the FREE-FORM agent on identical (origin, horizon)
# cells. The free-form agent emits a price, not a signal, so its point forecast is
# converted to the implied signal it would have needed -- the same quantity
# signal_loc denotes -- making the two schemas directly comparable.
#
# Both free-form runs below read the SAME cached briefings as the anchored runs
# (predictor_id "..._news_cached_..."), so news is held fixed and only the schema
# and prompt wording vary.

FREEFORM_CACHED = {
    "preview": "agent_predictor_wti_analyst_news_cached_gemini-3.1-flash-lite-preview_continuous",
    "3.5-flash": "agent_predictor_wti_analyst_news_cached_gemini-3.5-flash_continuous",
}
ANCHORED_CACHED = {
    ("preview", "original"): "agent_predictor_wti_analyst_anchored_cached_news_gemini-3.1-flash-lite-preview_continuous",
    ("preview", "symloc"): "agent_predictor_wti_analyst_anchored_cached_news_symloc_gemini-3.1-flash-lite-preview_continuous",
    ("preview", "twosided"): "agent_predictor_wti_analyst_anchored_cached_news_twosided_gemini-3.1-flash-lite-preview_continuous",
    ("3.5-flash", "original"): "agent_predictor_wti_analyst_anchored_cached_news_gemini-3.5-flash_continuous",
    ("3.5-flash", "symloc"): "agent_predictor_wti_analyst_anchored_cached_news_symloc_gemini-3.5-flash_continuous",
    ("3.5-flash", "twosided"): "agent_predictor_wti_analyst_anchored_cached_news_twosided_gemini-3.5-flash_continuous",
}
_offset_2b = pd.tseries.frequencies.to_offset(task.frequency)


def implied_signals(predictor_id: str) -> dict[tuple, float]:
    """Map (origin, horizon) -> signal_loc, backing it out of a price when not logged."""
    results = load_multi_backtest_results(eval_spec, predictor_id)
    if results is None:
        return {}
    out: dict[tuple, float] = {}
    for pred in results[task.task_id].predictions:
        as_of = pd.Timestamp(pred.as_of)
        horizon = {(as_of + _offset_2b * h): h for h in task.horizons}.get(pd.Timestamp(pred.forecast_date))
        if horizon is None:
            continue
        try:
            anchor = anchor_source.get(as_of=pred.as_of, horizon=horizon)
        except KeyError:
            continue  # anchor-table horizon gap, same one flagged in section 2
        signal = pred.metadata.get("signal_loc")
        if signal is None:
            signal = (pred.payload.point_forecast - anchor.point_forecast) / anchor.half_width
        out[(as_of, horizon)] = signal
    return out


rows_2b = []
for model, ff_id in FREEFORM_CACHED.items():
    free_form = implied_signals(ff_id)
    for prompt in ("original", "symloc", "twosided"):
        anchored = implied_signals(ANCHORED_CACHED[(model, prompt)])
        shared = sorted(set(free_form) & set(anchored))
        x = np.array([free_form[k] for k in shared])
        y = np.array([anchored[k] for k in shared])
        up, down = x > 0, x < 0
        rows_2b.append({
            "Model": model,
            "Prompt": prompt,
            "n": len(shared),
            "pass-through, free-form UP": y[up].mean() / x[up].mean(),
            "pass-through, free-form DOWN": y[down].mean() / x[down].mean(),
            "anchored mean where free-form said DOWN": y[down].mean(),
            "followed down (of n down)": f"{int((y[down] < 0).sum())} / {int(down.sum())}",
        })

df_rectification = pd.DataFrame(rows_2b).set_index(["Model", "Prompt"])
print("Pure shrinkage predicts the two pass-through ratios MATCH. A positive")
print("'anchored mean where free-form said DOWN' means the signal was flipped.\n")
df_rectification

Pure shrinkage predicts the two pass-through ratios MATCH. A positive
'anchored mean where free-form said DOWN' means the signal was flipped.



n  pass-through, free-form UP  \
Model     Prompt                                     
preview   original  50                       0.589   
          symloc    50                       0.598   
          twosided  50                       0.700   
3.5-flash original  50                       0.993   
          symloc    50                       0.836   
          twosided  50                       0.021   

                    pass-through, free-form DOWN  \
Model     Prompt                                   
preview   original                        -0.071   
          symloc                           0.291   
          twosided                         0.372   
3.5-flash original                         0.522   
          symloc                           0.996   
          twosided                         0.801   

                    anchored mean where free-form said DOWN  \
Model     Prompt                                              
preview   original                                    0.022   
          symloc                                     -0.091   
          twosided                                   -0.116   
3.5-flash original                                   -0.108   
          symloc                                     -0.206   
          twosided                                   -0.166   

                   followed down (of n down)  
Model     Prompt                              
preview   original                    9 / 25  
          symloc                     12 / 25  
          twosided                   17 / 25  
3.5-flash original                   18 / 28  
          symloc                     22 / 28  
          twosided                   23 / 28

### What the two prompt changes did — one framing fix, one structural fix

**`symloc` (framing only): partial fix, model-dependent.** 3.5-flash's down-side
pass-through went `-0.08` → `+0.31` — correctly signed for the first time, and it
followed the free-form agent down in **13 of 21** cells (up from 9). preview barely
moved: `-0.37` → `-0.17`, still wrong-signed, still following down in only **1 of
14**. Rewording alone reaches the model that was already close to reporting
negative signals and does little for the one that wasn't.

**`twosided` (structural — forced bearish case + de-loaded vocabulary): fixes what
`symloc` couldn't.** For **preview**, down-side pass-through goes `-0.37`
(original) → `-0.17` (symloc) → **`+0.32`** (twosided) — sign-correct for the
first time, and it now follows the free-form agent down in **8 of 14** cells,
up from 1. That directly overturns the read after `symloc` alone: preview's
inability to report a negative deviation was **not** a hard capability limit —
it responded once the rewrite went beyond wording (forcing a written bearish
case, stripping the one-directional "geopolitical risk" framing) rather than
just restating the existing bullets more plainly. **For 3.5-flash**, `twosided`
pushes down-side pass-through even further positive (`+0.40`, the best of the
three) but *also* pulls up-side pass-through down hard, `1.15` → `0.47` — the
structural change doesn't just unlock the negative side, it compresses the
positive side too. Its two ratios (`0.47` up vs. `0.40` down) end up the closest
to genuinely symmetric of any variant tested, at the cost of shrinking the
signal's magnitude overall.

**CRPS on the real 18-origin run (preview, `w_loc=0.2`) doesn't reward either
fix**: `original` 8.972 → `symloc` 9.016 → `twosided` 9.072, essentially flat to
slightly worse, and 80% coverage actually drops on `twosided` (42.9% → 38.1%,
still well above the raw baselines' 24%). This is the same point made below the
original scorecard: **the scorecard is the wrong instrument for this defect.**
Both rewrites make the signal's *sign* more honest — `twosided` visibly more so
— without that showing up as a CRPS win, because `w_loc=0.2` compresses whatever
arrives regardless of its sign. The paired table above, not the scorecard, is
the right instrument for judging these changes, and it should be re-run against
any future change to the location mechanism.

> **Consequence for section 8.** The ablation there is computed from the
> *original* prompt's logged signals, whose negative half is largely missing.
> In a window where the truth sat above the anchor 60% of the time, a
> rectified-positive signal is flattered. Read "location-only beats anchor-only"
> as *not yet established* rather than as a demonstrated property of the
> location mechanism — and note that even `twosided`, the more successful fix,
> only partially closes the gap for 3.5-flash and doesn't close it symmetrically.</cell id="a953d284">

## 3. Scorecard — mean CRPS, MAE(h=21), 80% coverage

In [7]:
scorecard_rows = []
for name, results in eval_results.items():
    scores = score_backtest_results(results, data_service)
    scorecard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS (2026)": scores.get("mean_crps", float("nan")),
            "MAE h=21d (2026)": scores.get("mae_h21", float("nan")),
            "80% CI Coverage": scores.get("coverage_80", float("nan")),
        }
    )
df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor").sort_values("Mean CRPS (2026)")
df_scorecard

,Mean CRPS (2026),MAE h=21d (2026),80% CI Coverage
Predictor,,,
"News Agent (3.5-flash, LIVE search baseline)",8.028,10.795,46.000
"News Agent (preview, LIVE search baseline)",8.208,11.013,24.000
"Anchored Agent (real run, symloc prompt)",9.085,11.777,44.000
"Anchored Agent (real run, twosided prompt)",9.088,11.831,38.000
Anchored Agent (real run),9.110,11.838,42.000
AutoARIMA (seeded anchor),9.264,11.788,30.000
"News Agent (3.5-flash, CACHED news)",9.437,12.234,44.000
Adaptive Agent (trained),9.603,12.318,40.909
"News Agent (preview, CACHED news)",9.721,12.049,30.000


In [8]:
eval_frame = predictions_to_frame(eval_results, data_service)
eval_board = leaderboard_with_uncertainty(eval_frame)
ph_crps = per_horizon_crps(eval_frame)
print(ph_crps.round(2).to_string())

                                              h=5d  h=8d  h=10d  h=15d  h=21d    All
predictor                                                                           
News Agent (3.5-flash, LIVE search baseline)  5.57   NaN   7.78    NaN  10.44   8.03
News Agent (preview, LIVE search baseline)    5.06   NaN   7.64    NaN  11.51   8.21
Anchored Agent (real run, symloc prompt)      6.59   NaN   8.30    NaN  12.00   9.08
Anchored Agent (real run, twosided prompt)    6.55   NaN   8.36    NaN  11.98   9.09
Anchored Agent (real run)                     6.55   NaN   8.27    NaN  12.13   9.11
AutoARIMA (seeded anchor)                     6.68   NaN   8.42    NaN  12.32   9.26
News Agent (3.5-flash, CACHED news)           6.23   NaN   8.63    NaN  13.01   9.44
Adaptive Agent (trained)                      5.41   NaN   7.38    NaN  15.22   9.60
News Agent (preview, CACHED news)             6.41   NaN   8.95    NaN  13.35   9.72
AutoARIMA                                     6.28   NaN  10.20  

## 4. Leaderboard with uncertainty — is any of this a real gap?

Error bars are ±1 standard error across the scored (origin, horizon) pairs. If the
bars of two methods overlap heavily, their ordering isn't statistically
distinguishable on 18 origins — the honest read, not just "who's on top".

In [9]:
viz.make_leaderboard_interval_chart(eval_board)

In [10]:
viz.make_crps_heatmap(ph_crps)

## 5. Per-origin panels through the shock

Realised price (black), point forecast (diamond) + 80% interval, one panel per
origin, ordered left→right through Feb–Jun 2026. Shows the real anchored agent
against the two raw free-form baselines it sits between in the scorecard.

The chart below has its own Plotly legend, but if it doesn't render in your
notebook viewer, the color key printed right after it is a plain-HTML fallback
that doesn't depend on Plotly's legend rendering at all.

In [11]:
_leaders = [
    "News Agent (3.5-flash, LIVE search baseline)",
    "News Agent (preview, LIVE search baseline)",
    "Anchored Agent (real run)",
    "Anchored Agent (real run, twosided prompt)",
]
print(f"Showing: {', '.join(_leaders)}")
_panel_fig = viz.make_eval_forecast_chart(eval_frame, price_df, _leaders)
_panel_fig.show()
display(HTML(color_key_html({"WTI history": viz.CLR_HISTORY, "Realised price": viz.CLR_ACTUAL,
                              **viz.predictor_colors(_leaders)})))

Showing: News Agent (3.5-flash, LIVE search baseline), News Agent (preview, LIVE search baseline), Anchored Agent (real run), Anchored Agent (real run, twosided prompt)


## 6. Verdict

**Current numbers (this notebook, as executed 2026-08-09, against the rebuilt
news cache + the anchor-gap fix — this is what to cite going forward):**
```
Mean CRPS (2026), best to worst:
  News Agent (3.5-flash, LIVE search baseline)    8.03   (n=50, 46% coverage)
  News Agent (preview, LIVE search baseline)      8.21   (n=50, 24% coverage)
  Anchored Agent (real run, symloc prompt)        9.09   (n=50, 44% coverage)
  Anchored Agent (real run, twosided prompt)      9.09   (n=50, 38% coverage)
  Anchored Agent (real run)                       9.11   (n=50, 42% coverage)
  AutoARIMA (seeded anchor)                       9.26   (n=50, 30% coverage)
  News Agent (3.5-flash, CACHED news)             9.44   (n=50, 44% coverage)
  Adaptive Agent (trained)                        9.60   (n=22, 41% coverage)
  News Agent (preview, CACHED news)               9.72   (n=50, 30% coverage)
  AutoARIMA                                      11.00   (n=22, 32% coverage)
  Naive (Last Value)                             13.64   (n=22,  0% coverage)
  Prophet                                        20.75   (n=16, 25% coverage)
```

**Two "News Agent" families, easy to conflate — read the label carefully.**
**LIVE search baseline** calls `search_web` for real, live, at eval time; it was
never touched by this session's news-cache rebuild and its numbers (8.03/8.21)
haven't moved. **CACHED news** reads the exact same `NewsCacheSource` briefings as
every anchored row above it — built specifically so the anchored-vs-free-form
comparison isn't confounded by the two agents reading different articles (see
section 2b). It's ~1.2-1.5 points worse than the live-search version, which is
expected and unrelated to news *quality* — live search sees a larger, freshly
retrieved set of articles per call; the cache is a fixed, pre-fetched snapshot.
Do not compare "beats/loses to the raw baseline" claims across the two families.

**Yes, it beats Prophet — by a lot.** Every method here, even plain AutoARIMA,
scores roughly 2-2.5x lower CRPS. Prophet's ~25% 2026 coverage (down from ~77% in
2025) is the same story `01_wti_case_study.ipynb` already tells qualitatively —
this notebook just puts a number on the generalization.

**The anchored agent does not beat the raw free-form baseline it's built on top
of** — comparing against the **LIVE search baseline**, the fair-on-methodology
but not-fair-on-shared-news comparison this notebook has always used for that
verdict. `Anchored Agent (real run)` scores 9.11 — worse than both raw `News
Agent` LIVE-search baselines (8.03 / 8.21). The `w_loc=0.2` fitted from the calm
2025 backtest costs CRPS in exactly the volatile regime the mechanism exists to
guard against. This agrees with the interview notes' n=14 dry-run direction on
this same eval window (looser scored better there too) and disagrees with the
2025 backtest's direction. **That tension is real** — see section 8's ablation
for what it does and doesn't mean about the mechanism itself. (Against the
**CACHED** free-form baseline instead — the apples-to-apples-on-news comparison —
the anchored agent actually wins: 9.11 vs. 9.72/9.44.)

**Coverage moved the right way**: 24% (raw baseline) → **42%** (anchored real
run) — closer to the 80% nominal target than either raw baseline manages. The
width mechanism is doing real, verifiable work even while the location mechanism
costs CRPS.

**Sample-size caveat — no longer applies.** See the resolved note in section 2:
the anchored row now resolves 50/54, essentially matching the raw baselines'
own count, so this is a fair comparison on sample size.

---

<details>
<summary><b>Historical, for the record — pre-2026-08-09 numbers (click to expand)</b></summary>

Before the news-cache rebuild + anchor-gap fix, this exact row read:
```
  Anchored Agent (real run)           8.97   (n=42, 43% coverage)
```
The anchored row got *worse* (8.97 → 9.11) once the fix restored the 8
previously-dropped cells (4 origins — 2026-02-02, 02-09, 05-11, 05-18 — that used
to fail outright). **The 8 new cells are not what drove the regression** — the
agent actually does *better* than its own average on them (mean CRPS 8.80 vs.
9.17 on the rest). The regression instead comes from the **other 42 cells**: the
same 14 origins that resolved before now score 9.17 on average, not 8.97,
because `force_refresh` reran everything against the rebuilt news cache
(different cutoff, different content) and LLM output isn't perfectly
deterministic even on unchanged input. See section 9 below — the 2026-03-02
shock origin alone explains most of that shift, and its story changed
substantially (new rationale, near-zero conviction instead of the old
high-conviction correct call — see the updated "one day" analysis).

Also, the CACHED News Agent rows above weren't loaded into this notebook's
`eval_results` at all before 2026-08-09 — this table only ever showed the LIVE
baseline. They were added specifically because comparing this notebook's LIVE
numbers against the separately-reported CACHED before/after
(`planning-docs/news-cache-rebuild-interview-notes.md` §11.2) caused confusion
about which "News Agent" was which.

</details></cell>


## 7. Langfuse traces — what was the agent actually thinking?

Every prediction from `AnchoredAgentPredictor` is stamped with a Langfuse trace
(`predict()` calls `stamp_forecast_on_trace` and logs `langfuse_trace_id`/
`langfuse_trace_url` into `Prediction.metadata`). `rationale` and `key_signals` are
also logged directly in the metadata — no Langfuse needed to read those. This is
the most direct answer to "why does it predict this well": the free text the agent
wrote down at prediction time, plus the full reasoning trace one click away.

In [12]:
anchored_rationales = extract_agent_rationales({"Anchored Agent (real run)": real_anchored_results})
print(f"{len(anchored_rationales)} rationale rows across {anchored_rationales['as_of'].nunique()} origins")
display(HTML(viz.render_rationales_html(anchored_rationales)))

50 rationale rows across 18 origins


## 8. Mechanism-isolation ablation — zero new LLM calls

`AnchoredAgentPredictor` already logs `signal_loc` and `signal_width` — the agent's
actual bounded judgment — into every prediction's metadata (see section 7's cards
above). Reconstructing `anchor-only`, `location-only`, and `scale-only` from those
*same logged signals*, just with different `(w_loc, w_width)` multipliers, holds
the agent's judgment fixed across every row and varies only how much of it the
reconstruction is allowed to use — cleaner than re-running the agent per row, which
would confound "different weights" with "different day's LLM variance", and free.

In [13]:
def reconstruct_ablation_row(base_result: BacktestResult, w_loc: float, w_width: float, label: str) -> BacktestResult:
    offset = pd.tseries.frequencies.to_offset(task.frequency)
    predictor_id = f"ablation_{label}"
    predictions: list[Prediction] = []
    scores: list[float] = []

    for pred in base_result.predictions:
        meta = pred.metadata
        as_of = pd.Timestamp(pred.as_of)
        expected = {(as_of + offset * h): h for h in task.horizons}
        horizon = expected.get(pd.Timestamp(pred.forecast_date))
        if horizon is None:
            continue
        anchor = anchor_source.get(as_of=pred.as_of, horizon=horizon)
        signal_loc = meta["signal_loc"]
        signal_width = meta["signal_width"]

        final_point = anchor.point_forecast + w_loc * signal_loc * anchor.half_width
        final_half_width = anchor.half_width * (1 + w_width * signal_width)
        scale = final_half_width / anchor.half_width
        final_q = {qq: final_point + (v - anchor.point_forecast) * scale for qq, v in anchor.quantiles.items()}

        actual = _resolve(pred.forecast_date)
        if actual is None:
            continue

        predictions.append(
            Prediction(
                predictor_id=predictor_id,
                task_id=task.task_id,
                issued_at=pred.issued_at,
                as_of=pred.as_of,
                forecast_date=pred.forecast_date,
                payload=ContinuousForecast(point_forecast=final_point, quantiles=final_q),
                metadata={"ablation_row": label, "w_loc": w_loc, "w_width": w_width,
                          "signal_loc": signal_loc, "signal_width": signal_width},
            )
        )
        scores.append(float(ps.crps_ensemble(actual, np.array(sorted(final_q.values())))))

    return BacktestResult(
        spec=single_spec, predictor_id=predictor_id, predictions=predictions, scores=scores,
        metric="crps", mean_score=float(np.mean(scores)) if scores else float("nan"),
        ran_at=_now, skipped_origins=0,
    )


ablation_results: dict[str, dict[str, BacktestResult]] = {
    "Ablation: Anchor-only (w_loc=0, w_width=0)": {task.task_id: reconstruct_ablation_row(_real, 0.0, 0.0, "anchor_only")},
    "Ablation: Location-only (w_loc=0.2, w_width=0)": {task.task_id: reconstruct_ablation_row(_real, W_LOC, 0.0, "location_only")},
    "Ablation: Scale-only (w_loc=0, w_width=0.5)": {task.task_id: reconstruct_ablation_row(_real, 0.0, W_WIDTH, "scale_only")},
    "Ablation: Full (w_loc=0.2, w_width=0.5)": {task.task_id: _real},
}

ablation_scorecard = []
for name, results in ablation_results.items():
    scores = score_backtest_results(results, data_service)
    ablation_scorecard.append({
        "Variant": name,
        "Mean CRPS": scores.get("mean_crps", float("nan")),
        "80% CI Coverage": scores.get("coverage_80", float("nan")),
    })
df_ablation = pd.DataFrame(ablation_scorecard).set_index("Variant")
df_ablation

,Mean CRPS,80% CI Coverage
Variant,,
"Ablation: Anchor-only (w_loc=0, w_width=0)",9.264,30.0
"Ablation: Location-only (w_loc=0.2, w_width=0)",9.283,26.0
"Ablation: Scale-only (w_loc=0, w_width=0.5)",9.088,40.0
"Ablation: Full (w_loc=0.2, w_width=0.5)",9.110,42.0


In [14]:
ablation_frame = predictions_to_frame(ablation_results, data_service)
ablation_board = leaderboard_with_uncertainty(ablation_frame)
viz.make_leaderboard_interval_chart(ablation_board)

### What the decomposition says

Both mechanisms help, and they roughly add up: `anchor-only` → `location-only` and
`anchor-only` → `scale-only` are each a real, individually-positive improvement
over the bare statistical anchor, and `full` (both together) improves on either
alone by roughly the sum of their individual gains — the two signals aren't
fighting each other or duplicating the same information, at least on this run.

This **reframes, rather than reverses, section 6's finding**. It isn't that the
bounded mechanism failed on 2026 — relative to the anchor it was built on top of,
both signals earned their keep. It's that `w_loc=0.2` (fit on the calm 2025
backtest) caps the location signal well below where the *unconstrained* free-form
baseline's own judgment went, and on this particular shock, the free-form agent's
larger, uncapped moves happened to pay off better than the capped version. The
2025-vs-2026 tension flagged in section 6 is really a tension about how tight the
cap should be, not about whether a location signal helps at all — this ablation is
evidence for the latter, not against it.

**On "why does it predict this well" (the rationale/Langfuse question in section
7)**: the `key_signals` and `rationale` fields show the agent citing concrete,
dated geopolitical developments (Hormuz-related strikes, OPEC/IEA demand-forecast
divergence) that a purely price-history model like the statistical anchor
structurally cannot see. That's the mechanism the location signal is trying to
bound, not manufacture — the news-reading capability is doing real work; the open
question is just how tightly that work should be capped.

## 9. Origin-by-origin: where does the raw baseline actually win?

Section 6 says the anchored agent scores worse than its raw free-form baseline on
average. Averages hide *where* — this section finds the specific origins driving
that gap, using `eval_frame` (already built in section 3) rather than any new
data source.

In [15]:
_compare = eval_frame[eval_frame["predictor"].isin(["Anchored Agent (real run)", "News Agent (preview, LIVE search baseline)"])]
_pivot = _compare.pivot_table(index=["as_of", "horizon"], columns="predictor", values="crps")
_pivot["gap"] = _pivot["Anchored Agent (real run)"] - _pivot["News Agent (preview, LIVE search baseline)"]
_pivot = _pivot.dropna()

_by_origin = _pivot.groupby(level="as_of")["gap"].sum().sort_values(ascending=False)
print("Per-origin total CRPS gap (anchored − baseline), summed across horizons — positive = anchored worse:")
print(_by_origin.round(2).to_string())

Per-origin total CRPS gap (anchored − baseline), summed across horizons — positive = anchored worse:
as_of
2026-03-02    61.02
2026-04-20    27.74
2026-04-06    14.51
2026-03-16     1.31
2026-03-23     0.82
2026-06-01     0.27
2026-02-09     0.25
2026-02-02    -1.36
2026-04-27    -2.01
2026-03-30    -2.18
2026-05-04    -2.47
2026-04-13    -3.34
2026-02-16    -5.15
2026-05-11    -5.36
2026-03-09    -6.76
2026-02-23    -6.76
2026-05-25   -11.87
2026-05-18   -13.54


One origin dominates everything else combined. Pull its detail — anchor,
reconstructed forecast, actual outcome, the agent's own `signal_loc`, and what it
wrote down at the time.

In [16]:
_worst_origin = _by_origin.index[0]
print(f"Worst origin: {_worst_origin.date()}\n")

_offset = pd.tseries.frequencies.to_offset(task.frequency)
_anchored = eval_results["Anchored Agent (real run)"][task.task_id]
_baseline = eval_results["News Agent (preview, LIVE search baseline)"][task.task_id]

for pred in _anchored.predictions:
    if pd.Timestamp(pred.as_of) != _worst_origin:
        continue
    _expected = {(pd.Timestamp(pred.as_of) + _offset * h): h for h in task.horizons}
    _h = _expected.get(pd.Timestamp(pred.forecast_date))
    _anchor = anchor_source.get(as_of=pred.as_of, horizon=_h)
    _actual = _resolve(pred.forecast_date)
    _m = pred.metadata
    _capped_move = W_LOC * _m["signal_loc"] * _anchor.half_width
    _wanted_move = _m["signal_loc"] * _anchor.half_width
    print(f"h={_h:>2}  anchor={_anchor.point_forecast:6.2f}  anchored_final={pred.payload.point_forecast:6.2f}  "
          f"actual={_actual:6.2f}   signal_loc={_m['signal_loc']:.2f} → wanted move {_wanted_move:+6.2f}, "
          f"capped to {_capped_move:+6.2f} by w_loc={W_LOC}")

print()
for pred in _baseline.predictions:
    if pd.Timestamp(pred.as_of) != _worst_origin:
        continue
    _actual = _resolve(pred.forecast_date)
    print(f"[raw baseline] point={pred.payload.point_forecast:6.2f}  actual={_actual:6.2f}")

_rationale = next(p.metadata.get("rationale") for p in _anchored.predictions if pd.Timestamp(p.as_of) == _worst_origin)
print(f"\nagent's rationale that day:\n  {_rationale}")

Worst origin: 2026-03-02

h= 5  anchor= 66.97  anchored_final= 67.25  actual= 94.77   signal_loc=0.20 → wanted move  +1.38, capped to  +0.28 by w_loc=0.2
h=10  anchor= 66.82  anchored_final= 67.01  actual= 93.50   signal_loc=0.10 → wanted move  +0.96, capped to  +0.19 by w_loc=0.2
h=21  anchor= 67.04  anchored_final= 67.04  actual=101.38   signal_loc=0.00 → wanted move  +0.00, capped to  +0.00 by w_loc=0.2

[raw baseline] point= 82.00  actual= 94.77
[raw baseline] point= 88.00  actual= 93.50
[raw baseline] point= 95.00  actual=101.38

agent's rationale that day:
  The market is currently reacting to a major supply-side data surprise (inventory build) while simultaneously pricing in significant geopolitical risk premiums related to Iran and the Red Sea. The immediate outlook is weighed down by the inventory report, but the upcoming OPEC+ policy determination and ongoing supply disruptions create high-variance risk, requiring a cautious approach to the statistical baseline.


### Does one day explain the whole gap?

In [17]:
_excl = _pivot[_pivot.index.get_level_values("as_of") != _worst_origin]
print(f"Mean CRPS, anchored vs. baseline — all {len(_pivot)} scored horizons:")
print(f"  anchored={_pivot['Anchored Agent (real run)'].mean():.3f}   baseline={_pivot['News Agent (preview, LIVE search baseline)'].mean():.3f}")
print(f"\nMean CRPS, excluding {_worst_origin.date()} ({len(_pivot) - len(_excl)} horizons removed):")
print(f"  anchored={_excl['Anchored Agent (real run)'].mean():.3f}   baseline={_excl['News Agent (preview, LIVE search baseline)'].mean():.3f}")

Mean CRPS, anchored vs. baseline — all 50 scored horizons:
  anchored=9.110   baseline=8.208

Mean CRPS, excluding 2026-03-02 (3 horizons removed):
  anchored=8.060   baseline=8.399


### Answer

**Current answer (2026-08-09, rebuilt cache, cutoff = 2026-02-27, three business
days before this origin):**

```
h= 5  anchor= 66.97  anchored_final= 67.25  actual= 94.77   signal_loc=0.20 → wanted +1.38, capped to +0.28
h=10  anchor= 66.82  anchored_final= 67.01  actual= 93.50   signal_loc=0.10 → wanted +0.96, capped to +0.19
h=21  anchor= 67.04  anchored_final= 67.04  actual=101.38   signal_loc=0.00 → wanted +0.00, capped to +0.00

Mean CRPS, anchored vs. baseline — all 50 scored horizons:
  anchored=9.110   baseline=8.208

Mean CRPS, excluding 2026-03-02 (3 horizons removed):
  anchored=8.060   baseline=8.399
```

**Excluding this one origin, the anchored agent now wins by 0.34 — not a tie.**
2026-03-02 alone (gap +61.02, out of the arm's total +45.1-point deficit vs. this
baseline — see section 9's per-origin table) fully explains why the anchored row
looks worse in the headline scorecard: strip out the one origin where the honest
signal correctly stayed near zero on a move nothing in the legitimate information
set could have anticipated, and the anchored mechanism is *ahead*, not tied and
not behind.

The rebuilt briefing for this origin has no mention of any shock at all — it
describes WTI "trading in the low-to-mid $60s," "consolidating in a bullish flag
pattern," with the U.S.-Iran talks (Feb 26) and the OPEC+ meeting (Mar 1) both
still framed as **upcoming, outcome unknown**. The agent's rationale: *"The
market is currently reacting to a major supply-side data surprise (inventory
build) while simultaneously pricing in significant geopolitical risk premiums
related to Iran and the Red Sea. The immediate outlook is weighed down by the
inventory report, but the upcoming OPEC+ policy determination and ongoing supply
disruptions create high-variance risk, requiring a cautious approach to the
statistical baseline."* — genuinely uncertain, not a shock call. This is the
**correct** epistemic state for someone who legitimately doesn't know the
strikes are coming yet — it costs more CRPS on this one lucky origin, but it's
honest.

**Revised framing.** The old "cap the location signal too tight" story is no
longer the right read of this origin. The question worth asking now is
different: is `w_loc=0.2` still costing CRPS on *genuinely* foreseeable moves
elsewhere, now that this origin's false-positive conviction is gone? Section 3's
cross-regime sign-flip (`anchor_regime_comparison.ipynb`) is the place to look
for that, not this one origin.

---

<details>
<summary><b>Historical, for the record — why this section's story changed (click to expand)</b></summary>

This is a rare case of directly observing a temporal-leakage fix change an
agent's actual decision, not just its aggregate score.

**Before (n=42, old `context/` cache, cutoff = origin day itself):** 2026-03-02
was "almost entirely one day" — `signal_loc` hit **0.75, 0.60, 0.40** (real,
high conviction), the agent's rationale explicitly named the weekend shock, and
excluding this one origin left the two methods "statistically tied": mean CRPS
7.79 (anchored) vs. 7.80 (baseline).

**Why the old conviction was real but illegitimate.** The old cached briefing
for this origin (`context/wti_news_2026-03-02.md`, cutoff = **2026-03-02**, the
same day) opens: *"As of March 1, 2026... On Sunday night, March 1, WTI prices
jumped over 8% to trade around $72 per barrel, up from approximately $67 per
barrel at Friday's close."* The agent is supposed to be forecasting **from**
Monday morning, March 2 — but its news briefing already reported Sunday night's
shock as a done deal. That "high-conviction correct call" was reading the
answer, not forecasting.

</details></cell>
